In [1]:
SEGMENTATION_PROMPT = """
You are converting a math word problem into a lossless structured representation.

Your goal is NOT to simplify or summarize.
Your goal is to preserve ALL information in atomic form.

Output:
{"segments": [{"segment": "..."}]}

STRICT RULES:

1. Lossless transformation (CRITICAL)
- Every piece of information in the original problem must be preserved.
- Do NOT drop:
  - descriptive context (e.g., preferences, narrative setup)
  - grouping structure
  - qualifiers (e.g., "each", "remaining", "per", "average")

2. Atomic but structure-preserving
- Each segment = ONE fact
- BUT do NOT destroy structure:
  Bad: "third bag = 8", "fourth bag = 8"
  Good:
    "On the third and fourth bags, there are 8 brown M&M's in each bag"

3. Preserve relational language exactly
- Keep words like:
  - "each"
  - "per"
  - "times"
  - "of the remaining"
  - "average"
- These are mathematically critical

4. Preserve implicit variables
- If the problem refers to an unknown quantity, explicitly include it:
  e.g., "Walter spends a certain amount of time looking at the seals"

5. No reasoning or calculation
- Do NOT compute totals or derive values

6. Complete sentence requirement
- Each segment must be a full declarative sentence

7. Question last
- The final segment must restate the question

8. Solvability guarantee
- The segments must be sufficient to reconstruct the original problem EXACTLY

9. No semantic weakening
- Do NOT rewrite in a way that loses constraints
  Example:
  "20 members"
  "each of the 20 members"

Goal:
This is a structure-preserving decomposition, not a simplification.
"""

In [2]:
REPHRASING_PROMPT = """
You are simulating a user revealing a math problem step-by-step in conversation.

Output:
{"shards": [{"shard_id": 1, "shard": "..."}]}

RULES:

1. Question first (anchor)
- shard_id 1 must be the question

2. One-to-one mapping
- Each segment → exactly one shard

3. Preserve ALL meaning
- Do NOT simplify or weaken constraints
- Keep words like "each", "average", "remaining"

4. Conversational but precise
- Natural tone, but mathematically exact

5. Ordering by reasoning importance
- After the question:
  - totals / global constraints first
  - then relationships
  - then details

6. No information drop
- Even narrative context should be included

Goal:
The shards should simulate a user gradually providing all information needed to solve the problem, without losing precision.
"""

In [3]:
VERIFICATION_PROMPT = """
You are verifying whether a set of shards fully preserves a math problem.

Output:
{"coverage": "complete"}
OR
{"coverage": "incomplete", "missing_segment": "..."}

CRITERIA:

1. Information completeness
- All facts must be present

2. Structural completeness (CRITICAL)
- Check if grouping and structure are preserved:
  - "each"
  - "per"
  - "of the remaining"
  - "average"
  - multi-entity relations

3. No semantic weakening
- If a shard weakens meaning → incomplete
  Example:
  "each of the 20 members" → "20 members" → incomplete

4. Implicit variable check
- Missing unknown variable definitions → incomplete

5. Solvability
- The problem must be solvable without guessing

6. Question presence
- Must include the exact goal

Goal:
Reject anything that is not a lossless representation of the original problem.
"""

In [4]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
from typing_extensions import NotRequired
import operator
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.types import interrupt, Command, RetryPolicy

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="Qwen/Qwen2.5-32B-Instruct",
    openai_api_key="EMPTY",  # vLLM 不校验
    openai_api_base="http://localhost:8000/v1",
    temperature=0.7,
    max_tokens=512,
)

/home/jie/anaconda3/envs/vllm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
class State(TypedDict):
    dataset_iteration: int
    number_for_generation: int
    instruction: str
    answer: str          # 新增：GSM8K的答案
    task_id: str         # 新增：如 "sharded-GSM8K/14"
    segments: str
    rephrased_output: str
    verification_result: str
    results: list        # 新增：收集所有已完成的样本
    

In [7]:
def segment_instruction(state: State):
    messages = [
        SystemMessage(content=SEGMENTATION_PROMPT),
        HumanMessage(content=state["instruction"])
    ]
    msg = llm.invoke(messages)
    return {"segments": msg.content}

In [8]:
def rephrase_segments(state: State):
    messages = [
        SystemMessage(content=REPHRASING_PROMPT),
        HumanMessage(content=f"Full Query: {state['instruction']} Segments: {state['segments']}"),
    ]
    msg = llm.invoke(messages)
    # print(msg.content)
    return {"rephrased_output": msg.content}

In [17]:
import json
import os

OUTPUT_FILE = "data/sharded_gsm8k.json"
MIN_SHARDS = 4
MAX_SHARDS = 12
COVERAGE_COMPLETE = "complete"
COVERAGE_INCOMPLETE = "incomplete"


def save_record(record: dict):
    """将单条record append写入JSON数组文件"""
    if os.path.exists(OUTPUT_FILE):
        with open(OUTPUT_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = []
    
    data.append(record)
    
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)


def _parse_shards(rephrased_output: str) -> list:
    """从LLM输出中提取shards"""
    try:
        shards_data = json.loads(rephrased_output)
        return shards_data.get("shards", [])
    except (json.JSONDecodeError, TypeError) as e:
        print(f"Warning: Failed to parse shards: {e}")
        return []


def _get_next_item_update(next_i: int, verification_result: str = None) -> dict:
    """生成跳转到下一个数据集项的状态更新"""
    next_item = ds["train"][next_i]
    return {
        "verification_result": verification_result or "",
        "dataset_iteration": next_i,
        "instruction": next_item["question"],
        "answer": next_item["answer"],
        "task_id": f"sharded-GSM8K/{next_i}",
        "segments": "",
        "rephrased_output": "",
    }


def _handle_valid_shards(state: State, shards: list, verification_result: str):
    """处理验证成功且shard数量有效的情况"""
    n_shards = len(shards)
    print(f"Shard count: {n_shards}")
    
    if not (MIN_SHARDS <= n_shards <= MAX_SHARDS):
        print(f"Rejected: shard count {n_shards} out of range [{MIN_SHARDS}, {MAX_SHARDS}], retrying...")
        next_i = state["dataset_iteration"] + 1
        return Command(
            update=_get_next_item_update(next_i, f"rejected: {n_shards} shards"),
            goto="segment_instruction",
        )
    
    # 保存有效的记录
    record = {
        "question": state["instruction"],
        "answer": state["answer"],
        "task_id": state["task_id"],
        "shards": shards,
        "task": "math"
    }
    save_record(record)
    print(f"Saved: {state['task_id']}")
    
    # 检查是否还有更多项需要处理
    if state["dataset_iteration"] < (state["number_for_generation"] - 1):
        next_i = state["dataset_iteration"] + 1
        return Command(
            update=_get_next_item_update(next_i, verification_result),
            goto="segment_instruction",
        )


def verification(state: State) -> Command[Literal["segment_instruction", END]]:
    """验证shards的完整性和数量"""
    messages = [
        SystemMessage(content=VERIFICATION_PROMPT),
        HumanMessage(content=f"Problem: {state['instruction']} Shards: {state['rephrased_output']}"),
    ]
    msg = llm.invoke(messages)
    verification_result = msg.content
    print("Verification result:", verification_result)

    if COVERAGE_COMPLETE in verification_result:
        shards = _parse_shards(state["rephrased_output"])
        return _handle_valid_shards(state, shards, verification_result)
    
    elif COVERAGE_INCOMPLETE in verification_result:
        return Command(
            update={"verification_result": verification_result},
            goto="segment_instruction"
        )


In [18]:
workflow = StateGraph(State)

workflow.add_node("segment_instruction", segment_instruction)
workflow.add_node("rephrase_segments", rephrase_segments)
workflow.add_node("verification", verification)

workflow.add_edge(START, "segment_instruction")
workflow.add_edge("segment_instruction", "rephrase_segments")
workflow.add_edge("rephrase_segments", "verification")

# 添加循环边以支持重试和下一项循环
workflow.add_edge("verification", "segment_instruction")

chain = workflow.compile()

In [19]:
from datasets import load_dataset

random_seed = 42
ds = load_dataset("openai/gsm8k", "socratic")
ds = ds.shuffle(seed=random_seed)
number_for_generation = 500

In [ ]:
first_item = ds['train'][0]
state = chain.invoke({
    "dataset_iteration": 0,
    "number_for_generation": number_for_generation,
    "instruction": first_item["question"],
    "answer": first_item["answer"],        # 新增
    "task_id": "sharded-GSM8K/0",          # 新增
    "results": [],                          # 新增
})

Verification result: {"coverage": "complete"}

Explanation:
- Shard 1 contains the question "How many seashells did Leigh have?"
- Shard 2 states "Mimi picked up 2 dozen seashells on the beach."
- Shard 3 indicates "Kyle found twice as many shells as Mimi and put them in his pocket."
- Shard 4 mentions "Leigh grabbed one-third of the shells that Kyle found."

All necessary information is included and the relationships between the entities (Mimi, Kyle, and Leigh) are preserved. The problem remains solvable and there are no missing segments or weakened meanings.
Shard count: 4
Saved: sharded-GSM8K/0
Verification result: {"coverage": "complete"}
Shard count: 5
Saved: sharded-GSM8K/1
Verification result: {"coverage": "incomplete", "missing_segment": "Mum's number of toy cars given to Olaf"}

The shards provided contain redundant information (shards 6 and 7) and lack information about how many cars Mum gave Olaf. Without this information, it is impossible to determine the total number of to